[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohsennasab/python-fundamentals-hh/blob/main/notebooks/09_naip-imagery/09_02_visualize_compare_naip.ipynb)

# Module 9, Lesson 2: Visualize and Compare NAIP Acquisitions
## True color, color infrared, and a careful look at what changed

### Welcome!

A drainage reviewer asks you a simple question: *"When did the subdivision south of the college go in, and can we use the older imagery as the pre-development condition?"* Two aerial images ten years apart look like they should answer it at a glance. Mostly they do. But put any two acquisitions side by side and you'll also see differences that have nothing to do with the land: a different sun angle, a different month, a different camera, a pond that's just a little lower that year.

In this lesson you'll bring the June 2012 and June 2022 NAIP images of the Lesson 1 teaching area onto a common grid, build true-color and color-infrared views, try NDVI as a visual aid, and write down what changed in a way you'd be comfortable putting in a memo. That means keeping **visible difference** and **confirmed change** in separate columns.

### What You'll Work Through Today

- Get both images from the fastest trustworthy source: your Lesson 1 output, the live service, or the course copy
- Compare their metadata side by side before comparing their pixels
- Confirm the band order before building any composite
- Build true-color and color-infrared (CIR) views
- Put both years on the same 1 m grid, and understand why a shared map extent isn't enough
- Compute NDVI and see why its values don't transfer between acquisitions
- Compare fixed patches and keep a structured observation log
- Find and fix a band-order bug that produces a convincing but wrong NDVI map

### Lesson Structure

1. **Mental Model** - Band order, composites, grids, and the difference between seeing and confirming
2. **Workspace Setup** - Packages and settings
3. **Get the Two Images** - Local, live, or course copy, all held to the same checks
4. **Check the Bands** - Before any composite
5. **True Color and Color Infrared** - Two ways to look at the same pixels
6. **A Common Analysis Grid** - Pixel-for-pixel comparability
7. **NDVI as a Visual Aid** - Useful, and easy to over-read
8. **Compare Fixed Patches** - A figure and an observation log
9. **Practice Debugging** - When NDVI looks convincing but is backwards

## Lesson Information

**Lesson Level:** Intermediate

**Purpose:** Compare two aerial acquisitions honestly: aligned, correctly labeled, and with visible differences kept separate from confirmed change.

### Learning Objectives

By the end of this lesson, you'll be able to:

- Read and verify a raster's band order, then build true-color and color-infrared composites from it.
- Explain the difference between source GSD, acquisition date, and output grid resolution.
- Resample two images onto a common grid with a method suited to continuous imagery, and verify the result.
- Compute NDVI from 8-bit NAIP bands and explain why NDVI values aren't directly comparable across acquisitions.
- Describe differences between two dates without calling them measured land cover change.

### Prerequisites

- Lesson 1 of this module, or at least its Part 1 mental model. You don't need to have run it; this notebook gets its own data.
- Module 4 raster concepts: CRS, transform, resolution, NoData

### Inputs and Outputs

| | File | Notes |
|---|---|---|
| Input | `naip_2022_teaching_aoi.tif` | From Lesson 1, the live service, or the course copy |
| Input | `naip_2012_teaching_aoi.tif` | From the live service or the course copy |
| Output | `module9_outputs/naip_2012_aoi_1m_common.tif`, `naip_2022_aoi_1m_common.tif` | Both years on one 1 m grid |
| Output | `module9_outputs/naip_two_year_views.png` | True color and CIR for both years |
| Output | `module9_outputs/naip_patch_comparison.png` | Side-by-side patches with dates and GSD |
| Output | `module9_outputs/naip_comparison_metadata.csv` | Source, date, GSD, and grid for each year |
| Output | `module9_outputs/naip_observation_log.csv` | Visible differences and possible explanations |

## Using AI in This Lesson

This lesson is full of pictures, and pictures are where it's easiest to fool yourself. Use your AI assistant as a second set of eyes on the *evidence*, not the image. Paste in band descriptions, grid checks, or NDVI summary tables and ask what they imply.

Questions worth trying:

- *"My 2012 image is 1 m and my 2022 image is 0.6 m. If I resample both to 1 m, which resampling method makes sense for continuous imagery, and why not nearest neighbor?"*
- *"Both images plot in the same place on my map. Does that mean their pixels line up? What would I check?"*
- *"The same athletic field has NDVI 0.27 in one year and 0.50 in another. List explanations that aren't 'the grass got healthier'."*

As in Lesson 1, don't paste signed URLs into a prompt. You won't need one here.

## Part 1: Mental Model

### Band Order Is Metadata, Not an Assumption

A four-band NAIP file is just four grids of numbers. Nothing in the numbers says "I'm red." The band order lives in metadata: the STAC item, the band descriptions in the GeoTIFF, and the product documentation. NAIP is Red, Green, Blue, Near infrared. Other products aren't. Landsat and Sentinel-2 start with coastal and blue bands, for example. Read the descriptions every time.

### Two Composites From the Same Pixels

| Composite | Bands shown as (red, green, blue) | What it's good for |
|---|---|---|
| True color | Red, Green, Blue | Looks like a photo; roofs, roads, soil |
| Color infrared (CIR) | NIR, Red, Green | Vegetation shows up bright red; water is near black; bare soil and pavement look gray, tan, or cyan |

CIR is a display choice. It doesn't create new information, but it makes the NIR band much easier to read.

### Same Map Extent Is Not the Same Grid

The 2012 image has 1 m pixels. The 2022 image has 0.6 m pixels, and its grid starts 0.4 m further west. Plot them both on a map and they'll land in the same place, but pixel (row 100, column 100) is a different piece of ground in each. Any pixel-by-pixel comparison needs both on **one grid**: same CRS, same origin, same cell size, same number of rows and columns.

We'll use a 1 m grid, the coarser of the two. Resampling the 1 m image down to 0.6 m wouldn't add detail. It would just make more copies of the same pixels.

### Seeing Isn't Confirming

Some differences between two acquisitions are real changes on the ground. Others come from:

- **Season and weather.** Grass greenness, pond levels, and shadows all shift from week to week.
- **Sensor and processing.** Different cameras, color balancing, and 8-bit scaling.
- **Resolution.** 0.6 m shows fence lines and small sheds that 1 m blurs together.
- **Position.** NAIP's horizontal accuracy spec allows a few meters of shift between years.

The goal of this lesson is an observation log that keeps those possibilities visible.

## Part 2: Workspace Setup

The install check and imports follow the same pattern as Lesson 1.

In [ ]:
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    'pystac_client': 'pystac-client',
    'planetary_computer': 'planetary-computer',
    'rasterio': 'rasterio',
    'geopandas': 'geopandas',
}
missing = [
    pip_name for module_name, pip_name in REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(module_name) is None
]
if missing:
    print(f"Installing: {', '.join(missing)}")
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
    print("Done.")
else:
    print("All required packages are already installed.")

In [ ]:
# Standard library
import datetime
import math
import shutil
from pathlib import Path

# Tables, arrays, and web requests
import numpy as np
import pandas as pd
import requests

# Vector and raster data
import geopandas as gpd
from shapely.geometry import box, shape
import rasterio
import rasterio.errors
from rasterio.enums import Resampling
from rasterio.plot import plotting_extent
from rasterio.transform import from_origin
from rasterio.warp import reproject
from rasterio.windows import Window, from_bounds

# NAIP catalog search and temporary access
import pystac_client
import planetary_computer
from pystac_client.stac_api_io import StacApiIO
from urllib3.util.retry import Retry

# Plotting
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.patches import Rectangle

# Every figure in this lesson uses Arial. Colab runs on Linux, which does not
# ship Arial, so the list falls back to Liberation Sans: a font drawn with the
# same letter widths and shapes, made as a drop-in replacement. Naming a font
# that does exist is also what keeps matplotlib from printing a warning for
# every label it draws.
FONT_PREFERENCE = ['Arial', 'Liberation Sans', 'Helvetica', 'DejaVu Sans']
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = FONT_PREFERENCE

installed_fonts = {font.name for font in font_manager.fontManager.ttflist}
active_font = next((name for name in FONT_PREFERENCE if name in installed_fonts), None)
print(f"Figure font: {active_font or 'matplotlib default'}")

for package in (np, pd, gpd, rasterio, pystac_client):
    print(f"{package.__name__:<15} {package.__version__}")

### Set the Lesson Controls

The AOI is the same as Lesson 1. The two years were checked against the catalog while this lesson was written: both have a single image that covers the whole AOI, both were flown in late June, and both have four bands. The notebook still validates every file it uses.

In [ ]:
# Same teaching AOI as Lesson 1 (NAD83 / UTM zone 13N meters)
AOI_CRS = 'EPSG:26913'
AOI_BOUNDS = (517750, 4549350, 518850, 4550900)   # west, south, east, north

# The two acquisitions to compare
COMPARISON_YEAR = 2012
REFERENCE_YEAR = 2022
YEARS = [COMPARISON_YEAR, REFERENCE_YEAR]

# Common analysis grid: the coarser of the two source GSDs
COMMON_PIXEL_SIZE = 1.0

BAND_NAMES = ('Red', 'Green', 'Blue', 'Near infrared')
STAC_URL = 'https://planetarycomputer.microsoft.com/api/stac/v1'
COURSE_DATA_URL = (
    'https://raw.githubusercontent.com/mohsennasab/python-fundamentals-hh/'
    'main/notebooks/09_naip-imagery/data'
)

OUTPUT_DIR = Path('module9_outputs')
OUTPUT_DIR.mkdir(exist_ok=True)
print(f"Comparing NAIP {COMPARISON_YEAR} with NAIP {REFERENCE_YEAR}")
print(f"Common grid: {COMMON_PIXEL_SIZE} m cells in {AOI_CRS}")

## Part 3: Get the Two Images

### Where the Images Come From

Each image comes from the first source that works, in this order:

1. **An existing local file** that passes validation. That's your Lesson 1 output if you ran it in this session, or the course data folder if you cloned the repository.
2. **The live Planetary Computer service**, using the same search, coverage check, and window read as Lesson 1.
3. **The course copy** on GitHub. It was made with the same code from the same NAIP items.

Every source goes through the same validator. A file isn't accepted just because it exists.

The helper cell is long, but most of it repeats Lesson 1. What's new is `validate_crop()`, which returns a metadata dictionary when a file passes and lists every problem when it doesn't, and `get_naip_crop()`, which sets the source order.

Two details keep this gentle on the free service. `open_naip_catalog()` is the same rate-limit-aware helper from Lesson 1, and `naip_items_for_aoi()` searches **once** and caches the result, because a single search already returns every year. Asking twice for what one request answers is how you reach a rate limit that didn't need to happen.

In [ ]:
TOKEN_MARKERS = ['sig=', 'se=', 'sp=', 'sv=', 'skoid=']


def validate_crop(path, year):
    """Return metadata for a NAIP teaching crop, or raise ValueError listing problems."""
    problems = []
    with rasterio.open(path) as source:
        tags = source.tags()
        bounds = source.bounds
        if source.count != 4 or any(dtype != 'uint8' for dtype in source.dtypes):
            problems.append(f"expected 4 uint8 bands, found {source.count} {source.dtypes}")
        if source.crs is None or source.crs.to_string() != AOI_CRS:
            problems.append(f"expected {AOI_CRS}, found {source.crs}")
        if tuple(source.descriptions) != BAND_NAMES:
            problems.append(f"unexpected band descriptions {source.descriptions}")
        if (bounds.left > AOI_BOUNDS[0] + 0.01 or bounds.bottom > AOI_BOUNDS[1] + 0.01
                or bounds.right < AOI_BOUNDS[2] - 0.01 or bounds.top < AOI_BOUNDS[3] - 0.01):
            problems.append("raster does not cover the AOI")
        if tags.get('naip_year') != str(year):
            problems.append(f"naip_year tag is {tags.get('naip_year')}, expected {year}")
        valid_fraction = (source.dataset_mask() > 0).mean()
        if valid_fraction < 0.999:
            problems.append(f"only {valid_fraction:.1%} of pixels are valid")
        if any(marker in ' '.join(tags.values()).lower() for marker in TOKEN_MARKERS):
            problems.append("an access token was found in the tags")
        metadata = {
            'naip_year': year,
            'acquisition_date': tags.get('acquisition_date'),
            'source_item_id': tags.get('source_item_id'),
            'source_gsd_m': float(tags.get('source_gsd_m', 'nan')),
            'pixel_size_m': round(source.res[0], 3),
            'width': source.width,
            'height': source.height,
            'crs': source.crs.to_string() if source.crs else None,
            'band_descriptions': ', '.join(d or '' for d in source.descriptions),
        }
    if problems:
        raise ValueError('; '.join(problems))
    return metadata


def read_aoi_window(item, bounds):
    """Read `bounds` from a NAIP item's image on its own pixel grid (as in Lesson 1)."""
    signed_href = planetary_computer.sign(item.assets['image'].href)
    with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR'):
        with rasterio.open(signed_href) as source:
            window = from_bounds(*bounds, transform=source.transform)
            col_start, row_start = math.floor(window.col_off), math.floor(window.row_off)
            col_stop = math.ceil(window.col_off + window.width)
            row_stop = math.ceil(window.row_off + window.height)
            if (col_start < 0 or row_start < 0
                    or col_stop > source.width or row_stop > source.height):
                raise RuntimeError("The AOI extends past the edge of this image.")
            window = Window(col_start, row_start, col_stop - col_start, row_stop - row_start)
            return source.read([1, 2, 3, 4], window=window), source.window_transform(window), source.crs


def write_naip_geotiff(path, bands, transform, crs, item):
    """Write a four-band NAIP crop with a mask and provenance tags (as in Lesson 1)."""
    valid = np.any(bands != 0, axis=0)
    profile = {'driver': 'GTiff', 'count': 4, 'dtype': 'uint8',
               'width': bands.shape[2], 'height': bands.shape[1],
               'crs': crs, 'transform': transform, 'compress': 'deflate',
               'predictor': 2, 'tiled': True, 'blockxsize': 256, 'blockysize': 256}
    with rasterio.Env(GDAL_TIFF_INTERNAL_MASK=True):
        with rasterio.open(path, 'w', **profile) as destination:
            destination.write(bands)
            destination.write_mask(valid.astype('uint8') * 255)
            for band_number, band_name in enumerate(BAND_NAMES, start=1):
                destination.set_band_description(band_number, band_name)
            destination.update_tags(
                source_program='USDA National Agriculture Imagery Program',
                source_host='Microsoft Planetary Computer', stac_collection='naip',
                source_item_id=item.id, naip_year=str(item.properties['naip:year']),
                acquisition_date=item.datetime.date().isoformat(),
                source_gsd_m=str(item.properties['gsd']),
                source_epsg=str(item.properties.get('proj:epsg') or item.properties.get('proj:code')),
                access_date=datetime.date.today().isoformat(),
                processing='AOI window read on the source pixel grid; no reprojection or resampling',
                license='Public domain (USDA Farm Service Agency)')


def was_rate_limited(error):
    """True if an exception looks like an HTTP 429 (see Lesson 1, Part 4)."""
    text = str(error).lower()
    return (getattr(error, 'status_code', None) == 429
            or 'rate limit' in text or '429' in text)


def open_naip_catalog():
    """Open the NAIP catalog with a session that waits out rate limits.

    Same helper as Lesson 1. Without `status_forcelist`, urllib3 retries
    dropped connections but not HTTP status codes, so one 429 ("too many
    requests") would end the search.
    """
    rate_limit_retry = Retry(
        total=5, backoff_factor=1.0, status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=None, respect_retry_after_header=True)
    return pystac_client.Client.open(
        STAC_URL, stac_io=StacApiIO(max_retries=rate_limit_retry))


# One search covers every year, so cache it rather than asking once per year
SEARCH_CACHE = {}


def naip_items_for_aoi():
    """Return every NAIP item intersecting the AOI, searching at most once.

    A failed search is remembered too. If the service is already rate limiting
    us, asking again for the second year would only add to the problem, so the
    remembered failure sends that year straight to the course copy.
    """
    if 'error' in SEARCH_CACHE:
        raise SEARCH_CACHE['error']
    if 'items' not in SEARCH_CACHE:
        lonlat = gpd.GeoSeries([box(*AOI_BOUNDS)], crs=AOI_CRS).to_crs('EPSG:4326').total_bounds
        try:
            catalog = open_naip_catalog()
            SEARCH_CACHE['items'] = list(catalog.search(
                collections=['naip'], bbox=list(lonlat)).items())
        except Exception as error:
            SEARCH_CACHE['error'] = error
            raise
    return SEARCH_CACHE['items']


def fetch_live_crop(year, destination):
    """Pick the item of `year` that covers the AOI and read the AOI window."""
    aoi_geometry = box(*AOI_BOUNDS)
    for item in naip_items_for_aoi():
        if int(item.properties['naip:year']) != year:
            continue
        footprint = gpd.GeoSeries([shape(item.geometry)], crs='EPSG:4326').to_crs(AOI_CRS).iloc[0]
        if footprint.covers(aoi_geometry):
            bands, transform, crs = read_aoi_window(item, AOI_BOUNDS)
            write_naip_geotiff(destination, bands, transform, crs, item)
            return
    raise RuntimeError(f"No {year} NAIP item covers the whole AOI.")


def get_naip_crop(year):
    """Return a validated crop for `year`: local file, then live service, then course copy."""
    filename = f'naip_{year}_teaching_aoi.tif'
    destination = OUTPUT_DIR / filename

    # Source 1: an existing local file
    for candidate in [destination, Path(filename), Path('data') / filename,
                      Path('notebooks/09_naip-imagery/data') / filename]:
        if candidate.exists():
            try:
                validate_crop(candidate, year)
                if candidate.resolve() != destination.resolve():
                    shutil.copyfile(candidate, destination)
                return destination, f"validated local file ({candidate})"
            except (ValueError, rasterio.errors.RasterioError) as error:
                print(f"  {candidate} was not usable: {error}")

    # Source 2: the live service
    try:
        fetch_live_crop(year, destination)
        validate_crop(destination, year)
        return destination, "live read from Planetary Computer"
    except Exception as error:   # any live-service problem falls through to the course copy
        print(f"  live read for {year} failed: {error}")
        if was_rate_limited(error):
            print("  (That's a rate limit, not a problem with your code. The course "
                  "copy below is identical.)")
        else:
            print("  (Service errors come from the server, not from your notebook.)")

    # Source 3: the course copy
    response = requests.get(f"{COURSE_DATA_URL}/{filename}", timeout=120)
    response.raise_for_status()
    destination.write_bytes(response.content)
    validate_crop(destination, year)
    return destination, "course repository download"


print("Helpers are ready. Nothing has been downloaded yet.")

### Run the Retrieval

Read the source line for each year. They don't need to match: one image might come from your Lesson 1 output and the other from the live service.

In [ ]:
crop_paths = {}
crop_sources = {}
for year in YEARS:
    print(f"NAIP {year}:")
    crop_paths[year], crop_sources[year] = get_naip_crop(year)
    print(f"  source: {crop_sources[year]}")
    print(f"  file:   {crop_paths[year]}")

### Compare Metadata Before Pixels

Before looking at any imagery, put the metadata for the two files side by side. This table is the first thing a reviewer should see in a comparison memo, because it shows every difference that could affect the images before anyone starts interpreting them.

In [ ]:
metadata = pd.DataFrame([validate_crop(crop_paths[year], year) for year in YEARS])
metadata['source_used'] = [crop_sources[year] for year in YEARS]
metadata.set_index('naip_year').T

What stands out:

- **Both were flown in late June**, June 23, 2012 and June 21, 2022. That's about as good a seasonal match as you'll get between NAIP years. Vegetation should be at a similar stage, though weather in the weeks before each flight still matters.
- **The source GSDs differ.** 1.0 m in 2012 and 0.6 m in 2022. The 2022 image has almost three times as many pixels for the same area.
- **Same CRS, different grids.** Both are EPSG:26913, but the widths and heights differ and, as you'll see in Part 6, so do the grid origins.

## Part 4: Check the Bands Before Any Composite

The next cell prints the band descriptions from each file and stops if they aren't what the composites below assume. Two lines of code, and they would have prevented a surprising number of wrong maps.

In [ ]:
for year in YEARS:
    with rasterio.open(crop_paths[year]) as source:
        print(f"NAIP {year} band descriptions: {source.descriptions}")
        if tuple(source.descriptions) != BAND_NAMES:
            raise RuntimeError(f"NAIP {year} band order is not {BAND_NAMES}. Stop and investigate.")

# Band positions used by every composite in this lesson (1-based, as Rasterio counts)
BAND = {name: number for number, name in enumerate(BAND_NAMES, start=1)}
print("\nBand numbers:", BAND)

From here on the notebook refers to bands by name through the `BAND` dictionary (`BAND['Near infrared']`) instead of typing `4`. If you ever load a product with a different order, you change one line.

## Part 5: True Color and Color Infrared

The next cell builds both composites for both years on their **native** grids and plots them in map coordinates. Each panel title carries the acquisition date and source GSD, so the figure explains itself if someone pastes it into a report.

The display stretch is calculated separately for each image (2nd to 98th percentile). That makes each image look its best, but it also hides overall brightness differences between the years. That's fine for visual review, and it's one reason we won't compare raw pixel values between years.

In [ ]:
def stretch_limits(bands, lower=2, upper=98):
    """Display limits (low, high) for each band of a (3, rows, cols) array."""
    return [np.percentile(band, [lower, upper]) for band in bands]


def percentile_stretch(bands, limits=None):
    """Stretch a (3, rows, cols) array to 0-1 for display. The data are not changed.

    Pass `limits` from a whole image when showing a small patch, so the patch
    isn't stretched on its own narrow range of values.
    """
    if limits is None:
        limits = stretch_limits(bands)
    display_array = np.zeros(bands.shape, dtype='float32')
    for index, (low, high) in enumerate(limits):
        band = bands[index].astype('float32')
        display_array[index] = np.clip((band - low) / max(high - low, 1), 0, 1)
    return np.moveaxis(display_array, 0, -1)


COMPOSITES = {
    'True color (R, G, B)': [BAND['Red'], BAND['Green'], BAND['Blue']],
    'Color infrared (NIR, R, G)': [BAND['Near infrared'], BAND['Red'], BAND['Green']],
}

fig, axes = plt.subplots(2, 2, figsize=(13, 17))
for row, year in enumerate(YEARS):
    info = metadata.set_index('naip_year').loc[year]
    with rasterio.open(crop_paths[year]) as source:
        extent = plotting_extent(source)
        for col, (label, band_numbers) in enumerate(COMPOSITES.items()):
            axis = axes[row, col]
            axis.imshow(percentile_stretch(source.read(band_numbers)), extent=extent)
            axis.set_title(f"NAIP {year} | {label}\nacquired {info['acquisition_date']}, "
                           f"source GSD {info['source_gsd_m']:g} m", fontsize=11)
            axis.set_xticks([])
            axis.set_yticks([])
plt.tight_layout()
views_path = OUTPUT_DIR / 'naip_two_year_views.png'
fig.savefig(views_path, dpi=130, bbox_inches='tight')
plt.show()
print(f"Saved: {views_path}")

Take a minute with this figure before reading on.

- **South half:** open rangeland in 2012; a subdivision, new streets, two roundabouts, and a large graded area in 2022. That's the change the reviewer asked about.
- **CIR, athletic field and campus lawns:** bright red in both years. Irrigated turf reflects strongly in NIR.
- **CIR, rangeland:** a dull tan or gray-green, not red. Cured late-June grass has much weaker NIR contrast than irrigated turf. It's still vegetation, just not *vigorous* vegetation.
- **CIR, ponds:** nearly black. Water absorbs NIR.
- **CIR, roofs and pavement:** gray, white, or cyan, depending on the material.

**Question to ask your AI assistant:** *"In my CIR image, the athletic field is bright red but the surrounding grassland is tan. Both are grass. What's the physical reason, and what does that mean if I try to map 'vegetation' from this image?"*

## Part 6: A Common Analysis Grid

### Why the Native Grids Don't Line Up

The next cell prints each image's affine transform. The first number is the pixel width and the third is the x coordinate of the upper-left corner. The 2022 image starts at 517,749.6 with 0.6 m pixels. The 2012 image starts at 517,750.0 with 1.0 m pixels. Column 500 lands on different ground in the two files.

In [ ]:
for year in YEARS:
    with rasterio.open(crop_paths[year]) as source:
        t = source.transform
        print(f"NAIP {year}: pixel {t.a:g} m, upper-left corner ({t.c:,.1f}, {t.f:,.1f}), "
              f"{source.width:,} x {source.height:,} pixels")
        x, y = rasterio.transform.xy(t, 0, 500)
        print(f"           column 500 of row 0 is centered at x = {x:,.1f}\n")

### Build the Grid and Resample Both Years

The common grid starts exactly at the AOI's upper-left corner and uses 1 m cells, so it's 1,100 columns by 1,550 rows.

Resampling method matters:

- **Average** combines all the source pixels that fall inside each 1 m cell, weighted by how much of the cell they cover. It's the right choice when going from 0.6 m to 1 m, because every source pixel contributes.
- **Bilinear** would sample only the four nearest source pixels, which can skip detail when coarsening.
- **Nearest** is for categorical rasters like land cover codes. For imagery it produces blocky, slightly shifted results.

We use average for both years. The valid-data mask is resampled separately with the minimum rule, so a 1 m cell counts as valid only if every source pixel feeding it was valid.

In [ ]:
common_width = round((AOI_BOUNDS[2] - AOI_BOUNDS[0]) / COMMON_PIXEL_SIZE)
common_height = round((AOI_BOUNDS[3] - AOI_BOUNDS[1]) / COMMON_PIXEL_SIZE)
common_transform = from_origin(AOI_BOUNDS[0], AOI_BOUNDS[3], COMMON_PIXEL_SIZE, COMMON_PIXEL_SIZE)


def to_common_grid(path):
    """Resample a four-band crop and its mask onto the common grid."""
    with rasterio.open(path) as source:
        destination = np.zeros((4, common_height, common_width), dtype='uint8')
        reproject(source=source.read(), destination=destination,
                  src_transform=source.transform, src_crs=source.crs,
                  dst_transform=common_transform, dst_crs=AOI_CRS,
                  resampling=Resampling.average)
        valid = np.zeros((common_height, common_width), dtype='uint8')
        reproject(source=(source.dataset_mask() > 0).astype('uint8'), destination=valid,
                  src_transform=source.transform, src_crs=source.crs,
                  dst_transform=common_transform, dst_crs=AOI_CRS,
                  resampling=Resampling.min)
        tags = source.tags()
    return destination, valid.astype(bool), tags


common = {}
common_paths = {}
for year in YEARS:
    bands, valid, tags = to_common_grid(crop_paths[year])
    common[year] = {'bands': bands, 'valid': valid}
    common_paths[year] = OUTPUT_DIR / f'naip_{year}_aoi_1m_common.tif'

    profile = {'driver': 'GTiff', 'count': 4, 'dtype': 'uint8',
               'width': common_width, 'height': common_height,
               'crs': AOI_CRS, 'transform': common_transform,
               'compress': 'deflate', 'predictor': 2, 'tiled': True}
    with rasterio.Env(GDAL_TIFF_INTERNAL_MASK=True):
        with rasterio.open(common_paths[year], 'w', **profile) as destination:
            destination.write(bands)
            destination.write_mask(valid.astype('uint8') * 255)
            for band_number, band_name in enumerate(BAND_NAMES, start=1):
                destination.set_band_description(band_number, band_name)
            destination.update_tags(**tags)
            destination.update_tags(
                output_pixel_size_m=str(COMMON_PIXEL_SIZE),
                processing=f"Resampled from source grid to common {COMMON_PIXEL_SIZE:g} m "
                           "grid with average resampling; mask resampled with min")
    print(f"Saved: {common_paths[year]}")

### Check the Grid

"Same grid" is a claim you can test. The next cell reopens both files and compares CRS, transform, and shape, then reports the valid fraction for each. If anything differs, it stops.

In [ ]:
grid_rows = []
for year in YEARS:
    with rasterio.open(common_paths[year]) as source:
        grid_rows.append({
            'naip_year': year,
            'crs': source.crs.to_string(),
            'pixel_size_m': source.res[0],
            'upper_left': (source.transform.c, source.transform.f),
            'width': source.width,
            'height': source.height,
            'valid_fraction': round(float((source.dataset_mask() > 0).mean()), 4),
            'source_gsd_m': float(source.tags()['source_gsd_m']),
        })
grid_check = pd.DataFrame(grid_rows)
display(grid_check)

same_grid = grid_check[['crs', 'pixel_size_m', 'upper_left', 'width', 'height']].nunique().eq(1).all()
if not same_grid:
    raise RuntimeError("The two common-grid files do not share one grid.")
print("Both years share one grid: same CRS, origin, pixel size, and dimensions.")

Notice the two resolution columns. `pixel_size_m` is 1.0 for both: that's the analysis grid. `source_gsd_m` is still 1.0 and 0.6. The 2022 image doesn't *become* 1 m imagery. It's 0.6 m imagery summarized onto 1 m cells, and both facts belong in your metadata.

### A Grid Match Isn't a Feature Match

Sharing a grid guarantees that cell (row, column) means the same coordinates in both files. It doesn't guarantee that the *ground features* were recorded in the same place. Each acquisition has its own positional error. The next figure zooms to a campus building that existed in both years and draws the same crosshair on each.

In [ ]:
# Display limits from each whole image, reused for every zoom and patch
display_limits = {year: stretch_limits(common[year]['bands'][:3]) for year in YEARS}

# An 80 m window around a campus building that exists in both years.
# The crosshair marks the building's northwest corner.
center_x, center_y, half = 518570, 4550245, 40
check_x, check_y = 518586, 4550262
row, col = rasterio.transform.rowcol(common_transform, center_x, center_y)
rows = slice(row - half, row + half)
cols = slice(col - half, col + half)
zoom_extent = (center_x - half, center_x + half, center_y - half, center_y + half)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
for axis, year in zip(axes, YEARS):
    patch = common[year]['bands'][[0, 1, 2], rows, cols]
    axis.imshow(percentile_stretch(patch, display_limits[year]), extent=zoom_extent,
                interpolation='nearest')
    axis.axvline(check_x, color='red', linewidth=1)
    axis.axhline(check_y, color='red', linewidth=1)
    axis.set_title(f"NAIP {year} on the common 1 m grid")
    axis.ticklabel_format(style='plain', useOffset=False)
    axis.tick_params(labelsize=8)
plt.tight_layout()
plt.show()

The building's west wall and north edge should sit within a pixel or two of the crosshair in both years. When this lesson was prepared, image matching over the campus put the offset between the two years at roughly 1 to 2 m, which is within NAIP's positional tolerance.

The practical rule: **a difference only a meter or two wide along an edge isn't evidence of change.** It might just be registration.

## Part 7: NDVI as a Visual Aid

### The Formula, and a Detail That Matters

The Normalized Difference Vegetation Index is

$$\text{NDVI} = \frac{\text{NIR} - \text{Red}}{\text{NIR} + \text{Red}}$$

It ranges from -1 to 1. Vigorous vegetation is high and water is strongly negative. With calibrated reflectance, soil and pavement usually sit near zero. With 8-bit NAIP, as you're about to see, pavement and dry grass often come out negative too.

NAIP bands are stored as `uint8`, whole numbers from 0 to 255. If you subtract two `uint8` arrays and the result would be negative, NumPy wraps around instead (for example, `10 - 20` becomes `246`). The helper converts to floating point *before* doing any arithmetic, and it leaves cells with `NIR + Red = 0` as NaN instead of dividing by zero.

In [ ]:
def compute_ndvi(bands, valid):
    """NDVI from a (4, rows, cols) R, G, B, NIR uint8 stack. Invalid cells are NaN."""
    red = bands[BAND['Red'] - 1].astype('float32')
    nir = bands[BAND['Near infrared'] - 1].astype('float32')
    total = nir + red
    ndvi = np.full(red.shape, np.nan, dtype='float32')
    np.divide(nir - red, total, out=ndvi, where=(total > 0) & valid)
    return ndvi


# The uint8 trap, in one line
print("uint8 10 - 20 =", np.array([10], dtype='uint8') - np.array([20], dtype='uint8'))

ndvi = {year: compute_ndvi(common[year]['bands'], common[year]['valid']) for year in YEARS}

fig, axes = plt.subplots(1, 2, figsize=(13, 9))
for axis, year in zip(axes, YEARS):
    image = axis.imshow(ndvi[year], cmap='RdYlGn', vmin=-0.5, vmax=0.6,
                        extent=(AOI_BOUNDS[0], AOI_BOUNDS[2], AOI_BOUNDS[1], AOI_BOUNDS[3]))
    axis.set_title(f"NDVI, NAIP {year}")
    axis.set_xticks([])
    axis.set_yticks([])
fig.colorbar(image, ax=axes, shrink=0.6, label='NDVI (same color scale for both years)')
plt.show()

### Same Field, Different Numbers

The maps look broadly similar: turf and trees are green, ponds and pavement are orange to red, and the rangeland is a pale tan just below zero. Now look at the *numbers* over a few fixed areas that didn't change use between the two years.

In [ ]:
REFERENCE_AREAS = {
    'Athletic field (irrigated turf)': (518180, 4550322, 518262, 4550425),
    'Rangeland north of campus': (517900, 4550580, 518000, 4550760),
    'North pond (open water)': (518488, 4550705, 518535, 4550750),
}

rows = []
for name, bounds in REFERENCE_AREAS.items():
    r0, c0 = rasterio.transform.rowcol(common_transform, bounds[0], bounds[3])
    r1, c1 = rasterio.transform.rowcol(common_transform, bounds[2], bounds[1])
    row = {'area': name}
    for year in YEARS:
        row[f'NDVI {year}'] = round(float(np.nanmean(ndvi[year][r0:r1, c0:c1])), 2)
    rows.append(row)
ndvi_reference = pd.DataFrame(rows)
ndvi_reference['difference'] = ndvi_reference[f'NDVI {REFERENCE_YEAR}'] - ndvi_reference[f'NDVI {COMPARISON_YEAR}']
ndvi_reference

The athletic field is the same irrigated turf in both years. When this lesson was prepared, its mean NDVI was about 0.27 in 2012 and about 0.50 in 2022. The pond's value shifted too, and the rangeland was slightly *negative* in both years.

None of that means the field got twice as healthy. NAIP is 8-bit imagery that's color-balanced for appearance, flown with different cameras in different years. NDVI computed from it is a **relative** index *within one image*. It's handy for seeing where vegetation is. It isn't a calibrated measurement you can subtract across years.

Two practical rules:

1. Use NDVI from NAIP to **see** vegetation patterns within one acquisition, not to measure vegetation health or change.
2. Don't subtract NDVI between NAIP years and call the result vegetation change.

**Question to ask your AI assistant:** *"Why would NDVI from 8-bit NAIP be negative over dry grassland? What would I need instead if I really did need comparable vegetation index values across years?"*

## Part 8: Compare Fixed Patches

### Choose Patches Before You Look

It's tempting to scan the whole image and circle whatever looks different. A more defensible habit is to decide which areas you'll compare *before* looking, based on the question you were asked. The reviewer asked about the subdivision, so that's one patch. We'll add the campus pond and field, where nothing should have changed, as a control, and the east construction area, which the drainage review will care about.

In [ ]:
PATCHES = {
    'A. West subdivision': (517950, 4549450, 518250, 4549750),
    'B. Campus pond and athletic field': (518030, 4550290, 518300, 4550500),
    'C. East construction ground': (518450, 4549650, 518850, 4549950),
}

fig, axes = plt.subplots(len(PATCHES), 2, figsize=(12, 5.2 * len(PATCHES)))
for row_index, (name, bounds) in enumerate(PATCHES.items()):
    r0, c0 = rasterio.transform.rowcol(common_transform, bounds[0], bounds[3])
    r1, c1 = rasterio.transform.rowcol(common_transform, bounds[2], bounds[1])
    patch_extent = (bounds[0], bounds[2], bounds[1], bounds[3])
    for col_index, year in enumerate(YEARS):
        info = metadata.set_index('naip_year').loc[year]
        patch = common[year]['bands'][[0, 1, 2], r0:r1, c0:c1]
        axis = axes[row_index, col_index]
        axis.imshow(percentile_stretch(patch, display_limits[year]), extent=patch_extent)
        axis.set_title(f"{name}\nNAIP {year}, acquired {info['acquisition_date']}, "
                       f"GSD {info['source_gsd_m']:g} m", fontsize=10)
        axis.set_xticks([])
        axis.set_yticks([])
plt.tight_layout()
patch_path = OUTPUT_DIR / 'naip_patch_comparison.png'
fig.savefig(patch_path, dpi=130, bbox_inches='tight')
plt.show()
print(f"Saved: {patch_path}")

### Keep an Observation Log

The log has separate columns for what you **see**, what **else** could explain it, and its **status**. The first three rows are filled in from this lesson's own review of the images, so you can see the level of detail that's useful. Your job is to check each one against the figure, edit anything you disagree with, and add at least one row of your own.

The `status` column is never "confirmed change" here. Confirming change takes evidence from outside the imagery: permits, plats, as-builts, a site visit, or a consistent, validated classification of both years.

In [ ]:
observations = pd.DataFrame([
    {
        'patch': 'A. West subdivision',
        f'observed_{COMPARISON_YEAR}': 'Open rangeland with faint two-track trails',
        f'observed_{REFERENCE_YEAR}': 'Curving residential streets, several dozen houses, lawns, driveways',
        'visible_difference': 'New residential development',
        'other_possible_explanations': 'None likely for streets and roofs; lawn color depends on irrigation and date',
        'status': 'Visible difference - confirm dates with plat or permit records',
    },
    {
        'patch': 'B. Campus pond and athletic field',
        f'observed_{COMPARISON_YEAR}': 'Two turf fields separated by a bare strip; pond open water larger along the south edge',
        f'observed_{REFERENCE_YEAR}': 'One continuous turf field; open water smaller, more vegetation at the south end',
        'visible_difference': 'Field layout changed; apparent change in open-water outline',
        'other_possible_explanations': 'Pond level varies by season and year; tree canopy can hide the water edge; 1-2 m registration offset',
        'status': 'Visible difference - water extent not confirmed',
    },
    {
        'patch': 'C. East construction ground',
        f'observed_{COMPARISON_YEAR}': 'Open rangeland',
        f'observed_{REFERENCE_YEAR}': 'Large graded area of bright bare soil, equipment, haul routes',
        'visible_difference': 'Active earthwork',
        'other_possible_explanations': 'Condition is temporary; it shows June 21, 2022 only',
        'status': 'Visible difference - temporary condition at acquisition date',
    },
])

# Add your own row here, then rerun the cell:
# observations.loc[len(observations)] = {...}

observations

### Export the Comparison Tables

Two CSVs travel with the figures: the metadata table, which says exactly which images were compared and how, and the observation log.

In [ ]:
metadata_export = metadata.copy()
metadata_export['common_grid_pixel_size_m'] = COMMON_PIXEL_SIZE
metadata_export['common_grid_file'] = [common_paths[year].name for year in YEARS]
metadata_export['resampling'] = 'average (mask: min)'
metadata_export['access_date'] = datetime.date.today().isoformat()

metadata_path = OUTPUT_DIR / 'naip_comparison_metadata.csv'
observations_path = OUTPUT_DIR / 'naip_observation_log.csv'
for table, path in [(metadata_export, metadata_path), (observations, observations_path)]:
    if any(marker in table.to_csv(index=False).lower() for marker in TOKEN_MARKERS):
        raise RuntimeError(f"Token-like text found; {path.name} was not saved.")
    table.to_csv(path, index=False)
    print(f"Saved: {path}")

## Part 9: Practice Debugging - A Convincing NDVI Map That's Backwards

A colleague sends you the cell below. They say they "remembered NAIP puts NIR first" and wrote it from memory. It runs, it makes a nice-looking map, and the colors even look plausible if you don't look too hard.

> **This cell contains an intentional bug.** It reads from your saved 2022 common-grid file but doesn't change it.

In [ ]:
# ---------------------------------------------------------------
# PRACTICE DEBUGGING: THIS CELL CONTAINS AN INTENTIONAL BUG.
# It runs without an error. Your job is to explain what's wrong.
# ---------------------------------------------------------------
with rasterio.open(common_paths[REFERENCE_YEAR]) as source:
    print("Band descriptions in this file:", source.descriptions)
    nir_band = source.read(1).astype('float32')   # "NAIP stores NIR first"
    red_band = source.read(4).astype('float32')
    rgb_check = source.read([1, 2, 3])

suspicious_ndvi = (nir_band - red_band) / np.maximum(nir_band + red_band, 1)

fig, axes = plt.subplots(1, 2, figsize=(13, 9))
axes[0].imshow(percentile_stretch(rgb_check))
axes[0].set_title(f"NAIP {REFERENCE_YEAR} true color (for reference)")
image = axes[1].imshow(suspicious_ndvi, cmap='RdYlGn', vmin=-0.6, vmax=0.5)
axes[1].set_title("Colleague's NDVI")
for axis in axes:
    axis.set_xticks([])
    axis.set_yticks([])
fig.colorbar(image, ax=axes[1], shrink=0.6)
plt.show()

for name, bounds in REFERENCE_AREAS.items():
    r0, c0 = rasterio.transform.rowcol(common_transform, bounds[0], bounds[3])
    r1, c1 = rasterio.transform.rowcol(common_transform, bounds[2], bounds[1])
    print(f"{name:<34} colleague's NDVI: {suspicious_ndvi[r0:r1, c0:c1].mean():+.2f}")

### Your Task

1. **Describe the symptom.** Compare the colleague's NDVI with the true-color image and with your Part 7 map. What's wrong with the athletic field? What about the ponds?
2. **Use the evidence, not memory.** The cell printed the band descriptions. Which band number holds NIR? Which holds red?
3. **Pick an independent check.** Before trusting a fix, name one feature where you *know* what NDVI should do, and check it. Hint: irrigated turf should be clearly positive, and open water should be negative.

**Try this with your AI assistant:**

*"This NDVI looks inconsistent with the true-color image: the irrigated field is negative and the pond is positive. Here are the printed band descriptions and the two `source.read()` lines. What's wrong, and what independent visual check would confirm the fix?"*

Write the corrected version below. A good fix uses the band descriptions (or the `BAND` dictionary) instead of hard-coded numbers.

In [ ]:
# YOUR FIX: rebuild the colleague's NDVI with the correct bands.
#
# Hints:
# 1. Look up band numbers with BAND['Near infrared'] and BAND['Red'].
# 2. Convert to float before subtracting.
# 3. Print the REFERENCE_AREAS means again. They should match Part 7.

## Engineering Interpretation and Limitations

### What This Comparison Supports

- A dated, side-by-side record showing that the subdivision and east earthwork appear between the June 2012 and June 2022 acquisitions
- A defensible pre-development visual reference for 2012, with its date and resolution stated
- A list of areas that need confirmation from permits, plats, or a site visit

### What It Doesn't Establish

- **When the change happened.** Anywhere between June 2012 and June 2022. Other NAIP years (2015, 2017, 2019) can narrow it.
- **How much land changed.** A visual comparison isn't an area measurement. Numeric change needs a consistent, validated classification of both years, which Lesson 3 starts on for one year.
- **Vegetation health or change from NDVI.** NAIP NDVI is relative within one image.
- **Water surface change.** Pond outlines vary with season, level, and canopy cover.

### Cautions

1. Always read band descriptions before building a composite or index.
2. Same map extent isn't the same grid. Resample to one grid before any pixel comparison.
3. Use average resampling when coarsening imagery; save nearest neighbor for categorical data.
4. Record both the source GSD and the analysis pixel size.
5. Keep "visible difference" and "confirmed change" in separate columns.

## Troubleshooting

| Problem | Likely cause | What to try |
|---|---|---|
| "You have exceeded a rate limit" (HTTP 429) | Too many anonymous requests from your address, often shared with other Colab users | Nothing to fix. The notebook retries, then uses the identical course copy. Rerun in a minute for the live path |
| Both years say "live read failed" | Planetary Computer busy or down | The notebook falls back to the course copy; rerun later for the live path |
| `validate_crop` rejects a local file | Wrong year, different AOI, or an older file | Delete the file in `module9_outputs` and rerun Part 3 |
| Grid check fails | A common-grid file was edited or from another run | Rerun Part 6 from the top |
| NDVI map is mostly one color | Integer math or wrong bands | Convert to float first; check `BAND` against the descriptions |
| NDVI values look "wrong" compared with published ranges | NAIP isn't calibrated reflectance | Compare patterns within one image, not numbers across products |
| CIR looks green instead of red for vegetation | Bands passed in the wrong order | CIR is `[NIR, Red, Green]` |
| Features shifted by a meter or two | Normal registration difference between years | Don't treat edge shifts of 1 to 2 m as change |

## Practice Exercises

### Exercise 1: Narrow Down the Construction Date

Lesson 1 found full-coverage imagery for 2015, 2017, and 2019 too. Change `COMPARISON_YEAR` to 2017, rerun Parts 3 to 8, and look at Patch A. Was the subdivision there yet? Try 2019. Record what you find as new rows in the observation log.

While you're at it, try 2015 and look at the rangeland. Is it the same color as in 2017? What would you write in the `other_possible_explanations` column?

The course copy only includes 2012 and 2022, so this exercise needs the live service.

In [ ]:
# EXERCISE 1: Narrow down when the subdivision appeared
# Your code here.
#
# Steps:
# 1. Set COMPARISON_YEAR = 2017 (then 2019) and rebuild YEARS.
# 2. Rerun get_naip_crop() and the common-grid step for the new year.
# 3. Look at Patch A. Add a row to the observation log with the date and GSD.

### Exercise 2: Add Your Own Patch

Pick a 250 m by 250 m area you find interesting. The north pond, the east campus parking lots, or the residential blocks on the west edge are all good choices. Add it to `PATCHES`, rerun the patch figure, and write an observation log row for it. Be strict about the `other_possible_explanations` column.

In [ ]:
# EXERCISE 2: Add a patch and an observation
# Your code here.
#
# Steps:
# 1. Add a new entry to PATCHES: name -> (west, south, east, north) in EPSG:26913.
# 2. Rerun the patch figure cell.
# 3. Add a matching row to the observations table and re-export it.

### Challenge: What Does a Half-Pixel Shift Look Like?

Make a deliberately wrong version of the 2012 common-grid image by building its destination transform from an origin shifted 0.5 m east. Then subtract the red band of the shifted and unshifted versions and plot the difference. Where do the largest differences show up, and what does that tell you about pixel-by-pixel change detection on imagery that isn't perfectly aligned?

Suggested prompt:

*"Help me create a copy of my 2012 NAIP image resampled onto a grid shifted by half a pixel, then map the difference from the correctly aligned version. Explain why the differences concentrate along edges."*

In [ ]:
# CHALLENGE: Half-pixel grid shift
# Your code here.
#
# Suggested pattern:
# shifted_transform = from_origin(AOI_BOUNDS[0] + 0.5, AOI_BOUNDS[3], COMMON_PIXEL_SIZE, COMMON_PIXEL_SIZE)
# ... reproject the 2012 crop onto shifted_transform, then compare band 1 ...

### Download Your Outputs

In Colab, the next cell downloads the lesson outputs through your browser. Locally it prints their paths.

In [ ]:
output_files = [*common_paths.values(), views_path, patch_path, metadata_path, observations_path]

try:
    from google.colab import files as colab_files
except ImportError:
    colab_files = None

for output_file in output_files:
    if not output_file.exists():
        raise FileNotFoundError(output_file)
    if colab_files is not None:
        colab_files.download(str(output_file))
    else:
        print(f"Output ready: {output_file.resolve()}")

## That's Lesson 2 Done!

You compared two aerial acquisitions ten years apart and did it carefully. You checked metadata before pixels, confirmed band order before building composites, resampled both years onto one verified grid, and saw firsthand that a meter or two of positional shift and a doubled NDVI value can both appear without anything changing on the ground. Your observation log answers the reviewer's question and says clearly what still needs confirming.

### Key Takeaways

- Band order is metadata. Read it, and refer to bands by name.
- Same map extent isn't the same grid. Resample to one grid, and verify it.
- Use average resampling to coarsen imagery. Record source GSD *and* analysis pixel size.
- NAIP NDVI is a within-image visual aid, not a calibrated measurement across years.
- Keep what you *see* separate from what you've *confirmed*.

### Next: Lesson 3

In `09_03_classify_naip_landcover.ipynb` you'll turn the 2022 image into a four-class screening map (vegetation, bare soil, built/paved, water) using a small labeled sample and a Random Forest, then check it against labels the model has never seen.

### Resources

- [Earth Data Science: NAIP multiband imagery in Python](https://earthdatascience.org/courses/use-data-open-source-python/multispectral-remote-sensing/intro-naip/)
- [GeoAI: data visualization examples](https://opengeoai.org/examples/data_visualization/)
- [NAIP on Microsoft Planetary Computer](https://planetarycomputer.microsoft.com/dataset/naip)
- [Rasterio: reprojection and resampling](https://rasterio.readthedocs.io/en/stable/topics/reproject.html)